[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C31_Coding_Agent_Course/04_edit_test_loop/04_edit_test_loop.ipynb)

# 04 · Edit-Test-Fix 循环（agent 的小脑）

目标：把前三模块的工具**编排**成一个会自我纠错的循环——**跑测试 → 解析 pytest 失败 → 定位 → 修复 → 再测**，直到测试绿或到步数上限。用 **MockLLM** 在 `tempfile` 玩具仓里**真跑 pytest**驱动整个循环，看着 bug 由红变绿。

路线：跑测试 → 解析 pytest(真实输出) → 失败定位 → MockLLM 驱动循环(端到端) → 收敛/防死循环→ ✏️ 练习 → 📖 答案 → 🧪 多 bug 真实胶囊。

> 心智模型：**agent 不靠一次写对，靠迭代逼近对。解析 pytest 文本→结构化信号是核心；程序管「测+判收敛」，LLM 管「看错+怎么改」。**

## 0 · 准备：复用前几模块的工具 + 一个带 bug 的玩具仓库

本模块要把工具**编排**起来，先把模块 01（文件）和 02（shell 跑 pytest）的精简版工具备好。

In [ ]:
import os, sys, re, tempfile, shutil, subprocess

WORK = tempfile.mkdtemp(prefix='c31_loop_')

# --- 模块 01：文件工具（精简）---
def read_file(work, path):
    return open(os.path.join(work, path), encoding='utf-8').read()
def edit_file(work, path, old, new):
    full = os.path.join(work, path); text = open(full, encoding='utf-8').read()
    cnt = text.count(old)
    if cnt != 1:
        raise ValueError(f'旧串匹配 {cnt} 处，需恰好 1 处')
    open(full, 'w', encoding='utf-8').write(text.replace(old, new))
    return f'已编辑 {path}'

# --- 模块 02：shell 跑 pytest（精简）---
# 注：--color=no 关掉 pytest 的 ANSI 颜色码，让输出能被正则干净解析（生产 agent 的标准做法）。
def run_tests(work, timeout=60):
    r = subprocess.run([sys.executable, '-m', 'pytest', '-q', '--color=no'],
                       cwd=work, capture_output=True, text=True, timeout=timeout,
                       env={**os.environ, 'PYTHONDONTWRITEBYTECODE': '1'})
    return (r.returncode == 0), (r.stdout + r.stderr)

# --- 带 bug 的玩具仓库 ---
def seed_buggy(work):
    open(os.path.join(work, 'calc.py'), 'w').write(
        'def add(a, b):\n    return a - b   # BUG: 应当是 a + b\n')
    open(os.path.join(work, 'test_calc.py'), 'w').write(
        'from calc import add\n\ndef test_add():\n    assert add(2, 3) == 5\n')
seed_buggy(WORK)
print('工具就绪 + 带 bug 玩具仓库就绪 ✅')

## 1 · 跑测试：拿到真实的 pytest 输出

先在玩具仓里**真跑** pytest，看它因 bug 而红，并打印它的**真实输出**——下一节要解析的就是它。

In [ ]:
passed, output = run_tests(WORK)
print('测试通过?', passed, '（False = 红）')
print('=== pytest 真实输出 ===')
print(output)
assert passed is False, 'bug 应让测试失败'
assert 'failed' in output.lower()
print('✅ 拿到真实 pytest 输出，下一节解析它')

## 2 · 解析 pytest 输出：文本 → 结构化信号 ⭐

从 pytest 的文本里抽三样：**总览**(几过几挂)、**失败用例**(哪个+原因)、**失败位置**(文件:行:错误类型)。
解析质量决定 LLM 下一步看到什么。

In [ ]:
def parse_pytest(output):
    # 防御：去掉可能残留的 ANSI 颜色码，保证 ^FAILED / file:line 正则能锚定行首
    output = re.sub(r'\x1b\[[0-9;]*m', '', output)
    m = re.search(r'(\d+) failed', output)
    n_failed = int(m.group(1)) if m else 0
    mp = re.search(r'(\d+) passed', output)
    n_passed = int(mp.group(1)) if mp else 0
    failed = re.findall(r'^FAILED (\S+)(?:\s+-\s+(.*))?$', output, re.M)
    locs = re.findall(r'^(\S+\.py):(\d+): (\w+)', output, re.M)
    return {'n_failed': n_failed, 'n_passed': n_passed,
            'failed_tests': [{'nodeid': t, 'message': msg} for t, msg in failed],
            'locations': [{'file': f, 'line': int(ln), 'error': e} for f, ln, e in locs]}

info = parse_pytest(output)
print('失败数:', info['n_failed'], '| 通过数:', info['n_passed'])
print('失败用例:', info['failed_tests'])
print('失败位置:', info['locations'])
assert info['n_failed'] == 1
assert any('test_add' in t['nodeid'] for t in info['failed_tests'])
assert any(l['file'].endswith('calc.py') and l['error'] == 'AssertionError'
           for l in info['locations']), '应解析出 calc.py 的 AssertionError 位置'
print('✅ 解析成功：把 pytest 文本变成了 LLM 能决策的结构化信号')

## 3 · 失败定位：区分症状(测试文件)与病灶(源码)

traceback 里的位置可能落在**测试文件**(症状：断言在哪挂)或**被测源码**(病灶：逻辑哪错)。
定位应优先指向**非测试**的源文件。

In [ ]:
def localize(info):
    '''从失败位置里挑出最该改的：优先非测试源文件。'''
    locs = info['locations']
    # 优先返回不在测试文件里的位置（病灶），否则退而返回任意位置（症状）
    source_locs = [l for l in locs if not os.path.basename(l['file']).startswith('test_')]
    chosen = source_locs[0] if source_locs else (locs[0] if locs else None)
    return chosen

target = localize(info)
print('定位到该改的位置:', target)
assert target is not None
assert target['file'].endswith('calc.py'), '应定位到被测源码 calc.py（病灶），而非测试文件'
# 看一眼那个文件，确认 bug 就在附近
print('\ncalc.py 内容:')
print(read_file(WORK, 'calc.py'))
print('✅ 定位正确：指向 calc.py（病灶），而非 test_calc.py（症状）')

## 4 · MockLLM 驱动 edit-test-fix 循环：端到端真跑 ⭐⭐

现在把一切缝起来。**MockLLM** 扮演 LLM 的修复决策（按脚本给出正确编辑），程序负责跑测试、解析、判收敛。
整个循环在玩具仓里**真跑 pytest、真改文件**，看着 bug 由红变绿。

In [ ]:
class FixMockLLM:
    '''确定性修复替身：按脚本依次给出 (old, new) 编辑指令。'''
    def __init__(self, script):
        self.script = list(script); self.i = 0
    def fix(self, info, work):
        '''真实 LLM 会读 info 决定怎么改；Mock 按脚本走。返回 (path, old, new)。'''
        if self.i >= len(self.script):
            return None
        action = self.script[self.i]; self.i += 1
        return action

def edit_test_fix(llm, work, max_iters=5, verbose=True):
    for i in range(max_iters):
        passed, output = run_tests(work)
        if passed:
            if verbose: print(f'  [第{i}轮] ✅ 测试全绿，收敛！')
            return {'success': True, 'iters': i}
        info = parse_pytest(output)
        if verbose: print(f'  [第{i}轮] 红：{info["n_failed"]} 个失败 → 解析+定位+修复')
        action = llm.fix(info, work)
        if action is None:
            return {'success': False, 'iters': i, 'reason': 'LLM 无更多动作'}
        path, old, new = action
        edit_file(work, path, old, new)               # 真的改文件
    return {'success': False, 'iters': max_iters, 'reason': '到步数上限'}

# MockLLM 脚本：第 1 轮就给出正确修复（a - b -> a + b）
seed_buggy(WORK)   # 重置成带 bug
llm = FixMockLLM([('calc.py', 'return a - b   # BUG: 应当是 a + b', 'return a + b')])
print('开始 edit-test-fix 循环：')
result = edit_test_fix(llm, WORK)
print('结果:', result)
assert result['success'] is True, '应当修复成功'
assert result['iters'] == 1, '第1轮改、第2轮验证绿'
assert 'return a + b' in read_file(WORK, 'calc.py')
print('✅✅ 端到端跑通：MockLLM 决策 + 真跑 pytest + 真改文件，bug 由红转绿！')

## 5 · 收敛判定与防死循环：步数上限 / 振荡 / 无进展

agent 可能永远改不对，必须有安全阀：**步数上限**(必备)、**振荡检测**(状态重复)、**无进展检测**(失败数不降)。

In [ ]:
import hashlib
def snapshot(work):
    '''给工作区所有 .py 文件内容算一个指纹（检测状态振荡）。'''
    h = hashlib.md5()
    for fn in sorted(os.listdir(work)):
        if fn.endswith('.py'):
            h.update(open(os.path.join(work, fn), 'rb').read())
    return h.hexdigest()

def edit_test_fix_guarded(llm, work, max_iters=8):
    seen, last_failed, stagnant = set(), None, 0
    for i in range(max_iters):
        passed, output = run_tests(work)
        if passed:
            return {'success': True, 'iters': i, 'reason': '测试全绿'}
        info = parse_pytest(output)
        if last_failed is not None and info['n_failed'] >= last_failed:
            stagnant += 1
        else:
            stagnant = 0
        if stagnant >= 3:
            return {'success': False, 'iters': i, 'reason': '连续3轮无进展'}
        last_failed = info['n_failed']
        fp = snapshot(work)
        if fp in seen:
            return {'success': False, 'iters': i, 'reason': '状态振荡'}
        seen.add(fp)
        action = llm.fix(info, work)
        if action is None:
            return {'success': False, 'iters': i, 'reason': 'LLM 无动作'}
        try:
            edit_file(work, *action)
        except ValueError:
            pass    # 编辑失败（如匹配不上）也算一轮，靠 guard 兜底
    return {'success': False, 'iters': max_iters, 'reason': '步数上限'}

# 演示防死循环：MockLLM 老是做无效编辑（改个无关紧要的注释，测试永不绿）
seed_buggy(WORK)
bad_llm = FixMockLLM([('calc.py', '# BUG: 应当是 a + b', '# still buggy %d' % k) for k in range(20)])
# 注意：每次改的是不同注释文本，状态指纹每轮不同 -> 靠「无进展」(失败数不降) 兜底
res = edit_test_fix_guarded(bad_llm, WORK, max_iters=8)
print('改不对时的结果:', res)
assert res['success'] is False, '一直改不对应当失败'
assert res['iters'] < 8, '应在到达上限前就被 无进展/振荡 提前止损'
print('✅ 防死循环生效：改不动时 agent 优雅放弃并报告原因，而非无限空转')

---
## ✏️ 练习 1：更全的 pytest 解析（含错误摘要与通过率）

增强解析：实现 `parse_full(output)` 返回 dict，含 `n_failed`、`n_passed`、`n_total`、`pass_rate`(0~1, 无测试时为 0.0)、`first_error`(第一个失败位置的 `'file:line: ErrorType'` 字符串，无则 `None`)。

In [ ]:
def parse_full(output):
    # TODO:
    #  n_failed = int(re.search(r'(\d+) failed', output).group(1)) if 命中 else 0
    #  n_passed 同理
    #  n_total = n_failed + n_passed
    #  pass_rate = n_passed / n_total if n_total else 0.0
    #  locs = re.findall(r'^(\S+\.py):(\d+): (\w+)', output, re.M)
    #  first_error = f'{f}:{ln}: {e}' (第一个 loc) 否则 None
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
_, out = run_tests(WORK) if False else (None, None)
# 用一段固定的样例输出测（也可换成真实 run_tests 的输出）
sample = ('F.\n=== FAILURES ===\n'
          'calc.py:2: AssertionError\n'
          'FAILED test_calc.py::test_add - assert 1 == 5\n'
          '1 failed, 1 passed in 0.05s\n')
info = parse_full(sample)
print(info)
assert info['n_failed'] == 1 and info['n_passed'] == 1 and info['n_total'] == 2
assert abs(info['pass_rate'] - 0.5) < 1e-9
assert info['first_error'] == 'calc.py:2: AssertionError'
# 全过的情况
allpass = parse_full('..\n2 passed in 0.01s\n')
assert allpass['n_failed'] == 0 and abs(allpass['pass_rate'] - 1.0) < 1e-9
assert allpass['first_error'] is None
print('✅ 练习 1 通过：解析出失败/通过/总数/通过率/首个错误位置')

## ✏️ 练习 2：失败定位优先级

实现 `pick_target(locations)`：从多个失败位置里选最该改的一个，优先级：
**非测试源文件 > 测试文件**；同类里取**行号最小**的。`locations` 为 `[{'file','line','error'}, ...]`，返回选中的 dict（空则 None）。

In [ ]:
def pick_target(locations):
    # TODO: 分成 source（basename 不以 test_ 开头）与 test 两组
    #       source 非空 -> 取其中 line 最小的；否则 test 非空 -> 取 line 最小的；都空 -> None
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
locs = [{'file':'test_calc.py','line':4,'error':'AssertionError'},
        {'file':'calc.py','line':10,'error':'TypeError'},
        {'file':'calc.py','line':2,'error':'AssertionError'}]
t = pick_target(locs)
print('选中:', t)
assert t['file'] == 'calc.py' and t['line'] == 2, '优先源文件、取行号最小'
# 只有测试文件时退而选它
only_test = pick_target([{'file':'test_x.py','line':9,'error':'AssertionError'}])
assert only_test['file'] == 'test_x.py'
assert pick_target([]) is None
print('✅ 练习 2 通过：源文件优先、行号最小，正确区分病灶与症状')

## ✏️ 练习 3：自己写一版 edit-test-fix 循环

实现 `my_loop(llm, work, max_iters)`：跑测试→绿则返回 `(True, i)`→否则解析后调 `llm.fix(info, work)` 拿 `(path,old,new)` 并 `edit_file`→到上限返回 `(False, max_iters)`。`llm.fix` 返回 None 时也应停止返回 `(False, i)`。

In [ ]:
def my_loop(llm, work, max_iters=5):
    # TODO: for i in range(max_iters):
    #         passed, output = run_tests(work)
    #         if passed: return (True, i)
    #         info = parse_pytest(output); action = llm.fix(info, work)
    #         if action is None: return (False, i)
    #         edit_file(work, *action)
    #       return (False, max_iters)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
seed_buggy(WORK)
llm = FixMockLLM([('calc.py', 'return a - b   # BUG: 应当是 a + b', 'return a + b')])
ok, iters = my_loop(llm, WORK, max_iters=5)
assert ok is True and iters == 1
assert 'return a + b' in read_file(WORK, 'calc.py')
# 多步修复：先改错一次(无效)，再改对
seed_buggy(WORK)
llm2 = FixMockLLM([('calc.py', '# BUG: 应当是 a + b', '# trying'),          # 第1轮：无效编辑
                   ('calc.py', 'return a - b', 'return a + b')])             # 第2轮：真修复
ok2, iters2 = my_loop(llm2, WORK, max_iters=5)
assert ok2 is True and iters2 == 2, '第1轮无效、第2轮修好、第3轮验证绿'
print(f'单步修复 iters={iters}; 两步修复 iters={iters2}')
print('✅ 练习 3 通过：自己的循环能驱动单步与多步修复直到收敛')

## ✏️ 练习 4：收敛/无进展判定

实现 `decide(history, max_iters)`：给定每轮失败数的历史列表 `history`（如 `[3,2,2,2]`），判断当前该
`'converged'`(最后一个为 0) / `'stagnant'`(最后 3 个相同且 >0) / `'budget'`(len≥max_iters) / `'continue'`(其余)。按此优先级返回字符串。

In [ ]:
def decide(history, max_iters=8):
    # TODO: 
    #  if history and history[-1]==0: return 'converged'
    #  if len(history)>=3 and history[-1]==history[-2]==history[-3] and history[-1]>0: return 'stagnant'
    #  if len(history)>=max_iters: return 'budget'
    #  return 'continue'
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
assert decide([3,2,1,0]) == 'converged'
assert decide([3,2,2,2]) == 'stagnant'       # 最后3个都是2
assert decide([5,4,3,2,1,1,1,1], max_iters=8) == 'stagnant'  # 无进展优先于 budget
assert decide([2,1], max_iters=2) == 'budget'
assert decide([3,2]) == 'continue'
assert decide([]) == 'continue'
# 收敛优先于一切
assert decide([0,0,0], max_iters=1) == 'converged'
print('✅ 练习 4 通过：能按 收敛>无进展>预算>继续 的优先级判定循环走向')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def parse_full(output):
    nf = re.search(r'(\d+) failed', output)
    npas = re.search(r'(\d+) passed', output)
    n_failed = int(nf.group(1)) if nf else 0
    n_passed = int(npas.group(1)) if npas else 0
    n_total = n_failed + n_passed
    pass_rate = n_passed / n_total if n_total else 0.0
    locs = re.findall(r'^(\S+\.py):(\d+): (\w+)', output, re.M)
    first_error = f'{locs[0][0]}:{locs[0][1]}: {locs[0][2]}' if locs else None
    return {'n_failed': n_failed, 'n_passed': n_passed, 'n_total': n_total,
            'pass_rate': pass_rate, 'first_error': first_error}

In [ ]:
# 练习 2 参考答案
def pick_target(locations):
    src = [l for l in locations if not os.path.basename(l['file']).startswith('test_')]
    tst = [l for l in locations if os.path.basename(l['file']).startswith('test_')]
    if src: return min(src, key=lambda l: l['line'])
    if tst: return min(tst, key=lambda l: l['line'])
    return None

In [ ]:
# 练习 3 参考答案
def my_loop(llm, work, max_iters=5):
    for i in range(max_iters):
        passed, output = run_tests(work)
        if passed:
            return (True, i)
        info = parse_pytest(output)
        action = llm.fix(info, work)
        if action is None:
            return (False, i)
        edit_file(work, *action)
    return (False, max_iters)

In [ ]:
# 练习 4 参考答案
def decide(history, max_iters=8):
    if history and history[-1] == 0:
        return 'converged'
    if len(history) >= 3 and history[-1] == history[-2] == history[-3] and history[-1] > 0:
        return 'stagnant'
    if len(history) >= max_iters:
        return 'budget'
    return 'continue'

---
## 🧪 真实数据胶囊：修一个有【两个】bug 的玩具仓库

把循环用在更真实的场景：一个模块里有**两个**独立 bug，对应两个失败测试。MockLLM 分两轮各修一个，循环**真跑 pytest** 直到两个都绿——这正是 agent 在 SWE 式任务里的迭代节奏。

In [ ]:
# 两个 bug 的玩具仓库
def seed_two_bugs(work):
    open(os.path.join(work, 'mathx.py'), 'w').write(
        'def add(a, b):\n    return a - b   # BUG1\n\n'
        'def mul(a, b):\n    return a + b   # BUG2\n')
    open(os.path.join(work, 'test_mathx.py'), 'w').write(
        'from mathx import add, mul\n\n'
        'def test_add():\n    assert add(2, 3) == 5\n\n'
        'def test_mul():\n    assert mul(2, 3) == 6\n')
seed_two_bugs(WORK)
passed0, out0 = run_tests(WORK)
info0 = parse_pytest(out0)
print('初始：失败数 =', info0['n_failed'], '（两个 bug）')
assert info0['n_failed'] == 2
print('✅ 胶囊准备就绪：两个独立 bug，对应两个失败测试')

**🧪 胶囊练习**：用 `FixMockLLM` 写一个**两步**修复脚本（先修 `add` 的 BUG1，再修 `mul` 的 BUG2），用 `edit_test_fix` 跑，断言**最终成功**且**用了 2 轮编辑**（第 3 轮验证全绿）。补全骨架。

In [ ]:
# TODO: 写一个两步修复脚本：
#  第1步: ('mathx.py', 'return a - b   # BUG1', 'return a + b')
#  第2步: ('mathx.py', 'return a + b   # BUG2', 'return a * b')
#  llm = FixMockLLM([...]) ; result = edit_test_fix(llm, WORK, max_iters=5)
raise NotImplementedError

In [ ]:
# 自测
assert result['success'] is True, '两个 bug 都应被修好'
assert result['iters'] == 2, '两轮编辑 + 第3轮验证全绿'
final = read_file(WORK, 'mathx.py')
assert 'return a + b' in final and 'return a * b' in final
passed_f, _ = run_tests(WORK)
assert passed_f is True, '最终 pytest 应全绿'
print('✅ 胶囊练习通过：循环分两轮修好两个 bug，真跑 pytest 确认全绿！')

In [ ]:
# 📖 胶囊参考答案
llm = FixMockLLM([
    ('mathx.py', 'return a - b   # BUG1', 'return a + b'),
    ('mathx.py', 'return a + b   # BUG2', 'return a * b'),
])
result = edit_test_fix(llm, WORK, max_iters=5)
print(result)

---
## 🔧 旁注：真实 LLM 怎么接进这个循环

把 MockLLM 换成真实 Claude，循环骨架**完全不变**——只是 `llm.fix(info, work)` 内部改成：

```python
def fix(self, info, work):
    # 1) 把结构化失败信号 + 相关代码 拼成 prompt
    prompt = (f"测试失败：{info['failed_tests']}\n"
              f"失败位置：{info['locations']}\n"
              f"相关代码：\n{read_file(work, info['locations'][0]['file'])}\n"
              f"请用 edit_file 工具修复【实现】（不要改测试来通过）。")
    # 2) 调 Claude（带 edit_file 工具）；它返回一个 tool_use(edit_file, {path, old, new})
    resp = self.client.messages.create(model='claude-sonnet-4-6',
                                       tools=[EDIT_TOOL_SCHEMA], messages=[...])
    # 3) 从 tool_use 块取出 (path, old, new) 返回
```

**关键纪律**：prompt 里明确「修实现、别改测试来通过」，防 reward hacking（见讲解第 7 节）。完整工具往返见模块 05。

In [ ]:
# 清理
shutil.rmtree(WORK, ignore_errors=True)
print('工作区已清理 ✅')

### 小结
- **循环骨架**：改→测→解析→定位→再改，直到绿或到上限；是模块 00 主循环的专门化（反馈控制）。
- **解析 pytest**：文本→结构化信号(总览/失败用例/位置)是技术核心；脆弱，优先解析最稳的部分 + 返回码兜底。
- **定位**：区分症状(测试文件)与病灶(源码)，优先改实现而非改测试。
- **收敛/防死循环**：步数上限(必备) + 振荡 + 无进展检测；会说「我搞不定」的 agent 胜过无限空转的。
- **测试质量与 reward hacking**：跑全套防回归；警惕 agent 改测试作弊——奖励可被操纵就失去意义。

下一站：**模块 05 · 完整编码 Agent** —— 把手/眼/脚/小脑组装成一个能端到端修 bug 的完整 agent。